# Session 2.1: Python Toolbelt for Data Stewards

Welcome back! In **Session 1** you set up JupyterLab, met the AI assistant [bia-bob](https://github.com/haesleinhuepf/bia-bob), and used it to *explore* a dataset and draft documents like a data analysis plan or a Jupyter notebook.

In **Session 2** we keep the same workbench and assistant, and introduce a small **toolbelt** of Python packages for everyday data steward tasks: validating, profiling, cleaning, documenting, and (optionally) depositing data.

> Throughout this session the rule is **AI-first, but always understand and verify**:  
> we let the assistant draft the code, then we *read it, run it, and check the result*.

## Learning objectives

By the end of Session 2 you will be able to:

1. Recognise common data quality problems in an example *raw incoming* dataset.
2. Define and enforce a **data schema** (a "data contract") with [`pandera`](https://pandera.readthedocs.io/).
3. Generate a shareable **data quality report** with [`fg-data-profiling`](https://github.com/Data-Centric-AI-Community/fg-data-profiling) (formerly named `ydata-profiling`).
4. Apply a few **light cleaning** steps and confirm they worked.
5. Describe and bundle your data into a documented, FAIR **data package** with [`frictionless`](https://framework.frictionlessdata.io/).
6. Draft a **DMP** section from what you found.
7. *(Optional)* Deposit a package to a repository ([Zenodo Sandbox](https://sandbox.zenodo.org/)) via its API.

## The data steward pipeline

The notebooks in this session follow one realistic workflow. Imagine a research project
hands you a raw data file. Your job is to turn it into something **validated, documented,
and ready to hand over or deposit**. So, our pipeline may look like this:

> EXPLORE → VALIDATE → REPORT & CLEAN → PACKAGE → DMP & DEPOSIT

| Notebook | Stage | Tool |
|---|---|---|
| `2_data-validation` | Validate | `pandera` |
| `3_quality-report` | Report & clean | `fg-data-profiling` |
| `4_metadata-packaging` | Describe & package | `frictionless` |
| `5_dmp-and-deposit` *(optional)* | DMP & deposit | assistant + Zenodo API |

Each tool is one item on the toolbelt. You don't need to memorise them - you need to know
**what each one is for** and how to ask the assistant to use it correctly.

## Initialize the AI assistant

This is the same initialization you saved at the end of Session 1. It reads your endpoint URL,
API key, and the **Data Steward system prompt** from your `secrets` file.

> Reminder: JupyterLab must have been started with your `secrets` file:  
> `uv run --env-file path/to/secrets jupyter lab`

In [ ]:
# Import os to read env vars
import os
# Import bob and helper functions
from bia_bob import bob, available_models, fix, doc

# Initialize assistant, reads API key from env var automatically
bob.initialize(
    # read endpoint URL from env var
    endpoint=os.getenv('ENDPOINT_URL'),
    # select an available model for your purpose
    model="mini",
    # read the Data Steward system prompt from env var
    system_prompt=os.getenv('SYSTEM_PROMPT_DATA_STEWARD'),
)

Let's confirm the assistant is available and in its Data Steward role:

In [2]:
%bob Briefly, in two sentences: what is your role, and what can you help me with ?

I’m an AI assistant that supports research data stewards by providing policy‑aware, practical guidance across the full data lifecycle—from planning and documentation to FAIR‑compliant sharing, preservation, and reuse. I can help you draft and review data management plans, design metadata and naming standards, conduct data profiling and quality checks, choose appropriate repositories, and create actionable checklists, templates, and documentation tailored to your institutional, funder, and domain requirements.

## Our working data: a raw incoming dataset

For this session we work with `../data/field_samples_raw.csv` - a small dataset of biological field samples that a project has handed over. It is *raw*: nobody has validated or cleaned it yet.

A **data dictionary** describing what the *clean* data should look like is provided in `../data/data_dictionary.md`. Open it now in JupyterLab (right-click > Open With > Markdown Preview) and keep it handy, it is our reference for "what good looks like".

Let's take a first look with [`pandas`](https://pandas.pydata.org/docs/). If `pandas` is not already available in your venv, you can use `uv` to install (add) it:

In [3]:
#!uv add pandas

If you do not understand what the following code does, you may ask your assistant ;)

In [4]:
import pandas as pd

df = pd.read_csv("../data/field_samples_raw.csv")

print("Shape (rows, columns):", df.shape)
df.head(10)

Shape (rows, columns): (182, 9)


,sample_id,species,island,date_collected,body_mass_g,flipper_length_mm,sex,region,notes
0,S0001,Adelie,Torgersen,2021-12-04,4547,201.2,.,Antarctica,juvenile?
1,S0002,Gentoo,NaN,2023-05-26,4650,212.4,Male,Antarctica,NaN
2,S0003,adelie,NaN,2022-10-09,2104,191.5,NaN,Antarctica,NaN
3,S0004,Chinstrap,Bisco,22/01/2021,5766,197.4,F,Antarctica,NaN
4,S0005,Chinstrap,Dream,2021-11-09,4814,219.9,F,Antarctica,NaN
5,S0006,Gentoo,Dream,2021-04-27,3959,203.0,.,Antarctica,collected near Pointe Géologie
6,S0007,Gentooo,NaN,5.8.2023,4168 g,182.6,F,Antarctica,NaN
7,S0008,Adelie,Dream,2021-01-28,4847 g,217.9,F,Antarctica,NaN
8,S0009,Gentoo,Torgersen,2021-11-24,3586,197.5,male,Antarctica,NaN
9,S0010,Adelie,Torgersen,4.9.2021,3667,199.6,M,Antarctica,NaN


**Take a moment to look.** Even from the first rows you can probably already spot trouble, e.g., in the `species`, `sex`, `body_mass_g`, or `date_collected` columns.

Let's ask the assistant for a quick first impression, using prompt augmentation via `{}`. However, always keep in mind the potential size of the augmented variable related to the available context size of the model behinf your assistant!

Notice we ask for a *short text answer, no code* - we just want orientation:

In [5]:
%%bob
Here is a raw dataset I just received: {df}
Give me a short, plain-language list of the most obvious data quality problems you notice.
Just a short bulleted text answer. No code, no cleaning yet.

- **Inconsistent species spelling/casing** (e.g., “Adelie”, “adelie”, “Gentoo”, “Gentooo”).
- **Missing values** in key columns such as `island`, `sex`, `body_mass_g`, and `notes`.
- **Mixed date formats** (`YYYY‑MM‑DD`, `DD/MM/YYYY`, `DD.MM.YYYY`) and ambiguous or malformed dates (e.g., “22/01/2021”, “28.3.2021”).
- **Placeholder or out‑of‑range values** (e.g., body mass of **‑999** as a sentinel for missing data).
- **Inconsistent sex codes** (e.g., “M”, “MALE”, “Male”, “F”, “.”, and missing values) and a dot placeholder.
- **Island names with typos or missing entries** (e.g., “NaN”, “Bisco” vs. “Biscoe”, “Dream”, “Torgersen”).
- **Notes column containing free‑text metadata** that may need standardisation (e.g., “weather: storm”, “juvenile?”).

Keep the assistant's list in mind, but remember to **verify** it against what you see. In the next notebooks we will systematically catch and fix these issues with the toolbelt, instead of eyeballing them.

## What's next

We continue with **`2_data-validation.ipynb`**, where we turn the data dictionary into an enforceable schema and let it tell us exactly which records break the rules.